### What Happens when Correlations Break Down?

Some simple simulations regarding factor covariances
___

Paul Grajzl

Friday, November 13



In [ ]:
import sys
sys.path.append("..")
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from src import simulation, plotting, factors

In [14]:
prices, returns, params = simulation.simulate_random_returns()

In [15]:
simulation.plot_simulated_stock(prices, params, "S0001")

### Part 1: Simulated Prices and Returns (Independent Stocks)

A baseline world with **no market, no factors, no structure**: every stock is an independent random walk.

**Model:** geometric Brownian motion. Each day, each stock's log return is

$$
\log\frac{P_{i,t}}{P_{i,t-1}} \sim \mathcal{N}\Big(\big(\mu_i - \tfrac{1}{2}\sigma_i^2\big)\Delta t,\ \sigma_i^2 \, \Delta t\Big), \qquad \Delta t = \tfrac{1}{252}
$$

The −½σ² term makes μ the expected annual simple return. Prices start at 100 and compound.

**Setup:**
- 1,000 stocks, 5 years of business days
- Each stock's true drift μ drawn uniformly from −10% to +20% a year (about a third negative)
- Each stock's true vol σ drawn uniformly from 15–60% a year
- Fixed random seed for reproducibility; true parameters stored for comparison

**Checks:**
- Realized vol matches true vol almost exactly (correlation ~0.99): 5 years of daily data pins down vol precisely
- Average pairwise correlation between stocks ≈ 0: no common driver
- Realized drift matches true drift **poorly**: 5 years is far too short to estimate expected returns. Risk is much easier to estimate than return.

In [18]:
caps, market, w0 = simulation.simulate_market_caps(prices)

# Top 10 stocks account for roughly 35% of the market concentration (real US market: top 10 roughly 30-40%)

simulation.plot_simulated_market(market, w0)

### Part 2: Market Caps and a Simulated Market Index

**Initial caps:** each stock gets a random share of a market worth 100 at the start, drawn from a **lognormal** distribution (skew parameter 1.75). This gives a right-skewed market like the real one: most stocks small, a handful very large. **Top 10 stocks ≈ 35% of initial market cap**, in line with the real US market (~30–40%).

**Dynamics:** share counts are fixed, so each stock's cap moves with its own price:

$$
\text{cap}_{i,t} = \text{cap}_{i,0} \times \frac{P_{i,t}}{P_{i,0}}
$$

**Market index:** the sum of all caps, starting at exactly 100. Its daily return is the cap-weighted average stock return, with weights drifting as stocks rise and fall, like a real index.

**Key observation:** the simulated market is far smoother than individual stocks. With independent stocks, their moves largely cancel out, so index vol comes out well below single-stock vols (15–60%). Real indices run ~15–20% vol because stocks share a common driver: the gap a market factor will close.

In [19]:
simulation.save_sim("baseline", prices=prices, returns=returns, params=params,
                    caps=caps, market=market, w0=w0)

In [ ]:
book = simulation.simulate_book_equity(caps)
bp = book / caps

In [23]:
# 4. One stock: daily drift with price, steps at each reporting date
simulation.plot_simulated_value(caps, book, "S0001")

In [24]:
simulation.save_sim("baseline", book=book)

### Part 3: Simulated Book Equity and Value

Value = **book-to-price** = book equity / market cap. In real life it moves for two reasons, on two clocks, and the simulation reproduces both:

1. **Price moves daily**: when a stock falls, its B/P rises (it gets cheaper), and vice versa
2. **Book updates quarterly, with a lag**: book grows with retained earnings, but the market only learns the new number when the company files, weeks after quarter end. Between filings, reported book is flat (a staircase)

**Construction (fake but realistic book equity):**
- **Starting B/P**: lognormal around a median of 0.4 (price-to-book ≈ 2.5, typical US market), wide dispersion (log sd 0.8), so both deep-value and expensive growth stocks exist
- **Correlation with size**: starting log B/P correlated with starting log market cap via a knob (default **−0.3**: big companies tend to be more expensive)
- **Book growth**: each company has its own long-run growth rate (mean 8% a year, sd 6%) plus quarterly noise (3%)
- **Point-in-time reporting**: each quarter's book becomes known only after a stock-specific lag of 20–45 days, on staggered fiscal calendars, so the cross-section doesn't all update on the same day

**Checks:**
- Median starting B/P ≈ 0.4; starting value–size correlation ≈ −0.3, by construction
- **Value–momentum correlation emerges on its own** (negative), never imposed: stocks that fell over the past year have low momentum *and* high B/P, because price sits in both. Same mechanism as in real markets
- Individual B/P paths wiggle daily with price and jump ~4 times a year at reporting dates

**Knobs for later experiments**: size correlation, valuation dispersion, book growth mean and dispersion.

**Known simplification**: book growth is unrelated to stock returns. In reality, earnings surprises move prices too; this link can be added deliberately if we want value to carry a genuine premium.

In [25]:
mkt_ret = market.pct_change()

expo_raw = {
    "Size": np.log(caps),
    "Value": np.log(book / caps),
    "Momentum": factors.momentum(prices),
    "Beta": factors.beta_exposure(returns, mkt_ret),
}
styles_sim = {k: factors.standardize(v, weights=caps) for k, v in expo_raw.items()}

In [26]:
res_sim, resid_sim = factors.cs_regression_multi(
    returns, styles_sim, weights=np.sqrt(caps), return_resid=True
)
res_sim["r2"].describe()

count    1007.000000
mean        0.016600
std         0.013973
min         0.000303
25%         0.007026
50%         0.012918
75%         0.021982
max         0.140246
Name: r2, dtype: float64

In [27]:
ones = pd.DataFrame(1.0, index=returns.index, columns=returns.columns)
expo_attr = {"intercept": ones, **styles_sim}

s1 = factors.attribute("S0001", returns, expo_attr, res_sim, resid_sim).rename(columns={"intercept": "Market"})

(s1.drop(columns="Total").sum(axis=1) - s1["Total"]).abs().max()     # exactness check, ~1e-16

plotting.plot_attribution(s1, "S0001")

In [28]:
plotting.plot_table(factors.risk_decomp(s1), title="S0001: risk decomposition (no-factor world)",
                    index_name="Component")

### Why It's All Idio in the Random-Walk World

### What "idio" means

A factor model explains a stock's return using only what it **shares with other stocks**:

$$
r_{i,t} = \underbrace{\sum_k x_{ik} \, f_{k,t}}_{\text{shared}} + \underbrace{\varepsilon_{i,t}}_{\text{idio}}
$$

The factor returns $f_{k,t}$ are estimated from the **whole cross-section** each day: they describe how groups of stocks moved together. Idio is whatever part of a stock's move **nobody else shared**.

### How this world was built

Every stock's daily return was drawn **independently**:

$$
r_{i,t} = \mu_i \Delta t + \sigma_i \sqrt{\Delta t} \, z_{i,t}, \qquad z_{i,t} \text{ independent across stocks}
$$

S0001's random shock on any day has nothing to do with any other stock's shock. So there is **nothing shared** for a factor to capture: by definition, every bit of its movement is idio.

### Why each factor fails

**Size, value, momentum, beta:** the characteristics exist (some stocks are big, some cheap), but returns were never generated from them. Each day the regression asks "did big stocks beat small ones today?" The answer is pure chance: a little positive one day, a little negative the next. So the estimated factor returns are noise, unrelated to S0001's actual move.

**Market:** the market here is just the cap-weighted average of independent stocks. S0001 is one of 1,000, with a small weight $w$, so its influence on the market is tiny. Its correlation with the market is roughly

$$
\rho \approx \frac{w \, \sigma_{\text{S0001}}}{\sigma_{\text{market}}}
$$

which is close to zero for a small weight. The market moves for reasons that are all about *other* stocks.

### Why contributions exist but don't count

The regression always outputs some factor return, so S0001's exposure × factor return is never exactly zero: its contributions have real-looking vol (6–7%). But risk share depends on **co-movement with S0001's total return**:

$$
\text{share}_j = \frac{\mathrm{Cov}(c_j, r)}{\mathrm{Var}(r)}
$$

Since those contributions are noise unrelated to S0001's moves, the covariance ≈ 0 and the share ≈ 0. All variance falls to idio.

### The takeaway

Factors explain a stock only to the extent its moves are **shared with other stocks that have similar characteristics**. Independent stocks share nothing, so everything is idio. Real stocks share a lot (the market, sectors, styles), which is why real idio shares were ~75% for Apple rather than ~100%.